# 01_data_sharing_agreement_<agreement> — governance control plane

Governance-owned, cross-environment working notebook for one agreement.

- Canonical notebook pattern: `01_data_sharing_agreement_<agreement>`.
- `%run 00_env_config` is runtime bootstrap only.
- AI suggestions are advisory; human approval is required before write-back.
- DQ enforcement stays in downstream `03_pc` notebooks.


In [ ]:
%run 00_env_config


In [ ]:
from IPython.display import HTML, display

from fabricops_kit import (
    prepare_business_context_profile_input,
    suggest_column_business_contexts,
    extract_column_business_context_suggestions,
    capture_column_business_context,
    COLUMN_BUSINESS_CONTEXT_FROM_WIDGET,
    write_column_business_context,
    build_governance_context,
    prepare_governance_input,
    suggest_pii_labels,
    extract_pii_suggestions,
    review_governance,
    write_governance,
    load_governance,
    load_approved_dq_rules,
    get_path,
    write_metadata_rows,
)


In [ ]:
# Agreement setup (lowercase placeholders)
agreement_id = "<agreement_id>"
agreement_name = "<agreement_name>"
agreement_context = "<agreement_context>"
approved_usage = "<approved_usage>"
business_context = "<business_context>"
ownership = "<ownership>"
permissions = "<permissions>"
restrictions = "<restrictions>"
classification = "<classification_posture>"
sensitivity = "<sensitivity_posture>"
pii = "<pii_posture>"
related_notebook_links = ["<related_notebook_link>"]

env_name = "dev"
dataset_name = "<agreement_name>"
table_name = "<table_name>"
metadata_path = get_path(env_name, "metadata", config=FABRIC_CONFIG)
save_to_metadata = False

governance_prompt_context = build_governance_context(
    business_context=business_context,
    approved_usage=approved_usage,
    dataset_context=agreement_context,
    steward_notes=f"Agreement={agreement_name}; ownership={ownership}; classification={classification}; sensitivity={sensitivity}; pii={pii}",
)

agreement_context_metadata = {
    "agreement_id": agreement_id,
    "agreement_context": agreement_context,
    "approved_usage": approved_usage,
    "business_context": business_context,
    "ownership": ownership,
    "permissions": permissions,
    "restrictions": restrictions,
    "classification": classification,
    "sensitivity": sensitivity,
    "pii": pii,
    "related_notebook_links": ", ".join(related_notebook_links),
    "agreement_name": agreement_name,
    "governance_prompt_context": governance_prompt_context,
}
print(f"Prepared agreement context for {agreement_id} ({table_name}).")


In [ ]:
# Save agreement metadata (governance-owned) to METADATA_DATA_AGREEMENT.
agreement_record = dict(agreement_context_metadata)
agreement_record.update({
    "environment_name": env_name,
    "dataset_name": dataset_name,
    "table_name": table_name,
})

if save_to_metadata:
    write_metadata_rows(
        spark,
        rows=[agreement_record],
        metadata_path=metadata_path,
        table_name="METADATA_DATA_AGREEMENT",
        mode="append",
    )
    print("Saved agreement metadata record.")
else:
    print("Dry run: skipped agreement metadata write.")


In [ ]:
# Fetch existing metadata with safe fallbacks.
def safe_load(table_name_to_read, *, filter_agreement=True, filter_table=False):
    try:
        df = spark.table(table_name_to_read)
        if filter_agreement and "agreement_id" in df.columns:
            df = df.filter(df.agreement_id == agreement_id)
        if filter_table and "table_name" in df.columns:
            df = df.filter(df.table_name == table_name)
        count = df.count()
        print(f"{table_name_to_read}: {count} rows")
        return df
    except Exception as exc:
        print(f"{table_name_to_read}: not available ({exc})")
        return None

profile_df = safe_load("METADATA_PROFILE_ROWS", filter_agreement=True, filter_table=True)
dq_rules_df = safe_load("METADATA_DQ_RULES", filter_agreement=True, filter_table=True)
column_context_df = safe_load("METADATA_COLUMN_CONTEXT", filter_agreement=True, filter_table=True)
column_governance_df = safe_load("METADATA_COLUMN_GOVERNANCE", filter_agreement=True, filter_table=True)
agreement_df = safe_load("METADATA_DATA_AGREEMENT", filter_agreement=True, filter_table=False)


In [ ]:
# Business context review from existing profile metadata.
if profile_df is not None and profile_df.count() > 0:
    profile_rows = [r.asDict(recursive=True) for r in profile_df.collect()]
    prepared_rows = prepare_business_context_profile_input(
        profile_rows,
        table_name=table_name,
        table_context=agreement_context,
    )
    prepared_df = spark.createDataFrame(prepared_rows)
    bc_response_df = suggest_column_business_contexts(prepared_df)
    bc_suggestions = extract_column_business_context_suggestions(bc_response_df)
    capture_column_business_context(
        bc_suggestions,
        environment_name=env_name,
        dataset_name=dataset_name,
        table_name=table_name,
    )
    print("Business context review widget launched.")
else:
    print("No profile metadata found. Populate METADATA_PROFILE_ROWS first.")


In [ ]:
# Save approved business context rows from widget state.
approved_business_context_rows = []
for row in COLUMN_BUSINESS_CONTEXT_FROM_WIDGET:
    enriched = dict(row)
    enriched.update({
        "agreement_id": agreement_id,
        "agreement_context": agreement_context,
        "approved_usage": approved_usage,
        "business_context": business_context,
        "ownership": ownership,
    })
    approved_business_context_rows.append(enriched)

if save_to_metadata and approved_business_context_rows:
    write_column_business_context(
        spark,
        rows=approved_business_context_rows,
        metadata_path=metadata_path,
        table_name="METADATA_COLUMN_CONTEXT",
        mode="append",
    )
    print(f"Saved approved business context rows: {len(approved_business_context_rows)}")
else:
    print("No approved business context rows saved (dry run or no approvals yet).")


In [ ]:
# Classification / sensitivity / PII review using approved business context.
if approved_business_context_rows:
    profile_rows = [r.asDict(recursive=True) for r in profile_df.collect()] if profile_df is not None else []
    prepared_governance_rows = prepare_governance_input(
        profile_rows,
        table_name=table_name,
        column_contexts=approved_business_context_rows,
    )
    governance_input_df = spark.createDataFrame(prepared_governance_rows)
    pii_response_df = suggest_pii_labels(governance_input_df)
    governance_suggestions = extract_pii_suggestions(pii_response_df)
else:
    governance_suggestions = []
    print("No approved business context rows available for governance suggestions.")

review_governance(
    suggestions=governance_suggestions,
    environment_name=env_name,
    dataset_name=dataset_name,
    table_name=table_name,
)
print("Governance review complete. Human approval is required before write-back.")


In [ ]:
# Save approved governance rows, preserving agreement_id in agreement_context_metadata.
if save_to_metadata:
    saved_rows = write_governance(
        spark,
        metadata_path=metadata_path,
        agreement_context=agreement_context_metadata,
        table_name="METADATA_COLUMN_GOVERNANCE",
        mode="append",
    )
    print(f"Saved governance rows: {len(saved_rows)}")
else:
    print("Dry run: skipped governance write.")


In [ ]:
# Existing DQ context (read-only visibility only).
if dq_rules_df is not None and dq_rules_df.count() > 0:
    approved_dq_rules = load_approved_dq_rules(dq_rules_df, table_name=table_name)
    print(f"Active approved DQ rules for {table_name}: {len(approved_dq_rules)}")
    for rule in approved_dq_rules[:10]:
        print(rule)
else:
    print("No METADATA_DQ_RULES rows found for this agreement/table.")


In [ ]:
# Notebook link registry for one agreement across environments/workspaces.
def render_notebook_registry(agreement_id_value: str):
    registry_df = safe_load("METADATA_NOTEBOOK_REGISTRY", filter_agreement=True, filter_table=False)
    if registry_df is None:
        registry_df = safe_load("METADATA_DATA_AGREEMENT", filter_agreement=True, filter_table=False)
    if registry_df is None or registry_df.count() == 0:
        print("No notebook registry found. Populate notebook_url / related_notebook_links metadata first.")
        return

    rows = [r.asDict(recursive=True) for r in registry_df.collect()]
    rows = [r for r in rows if str(r.get("agreement_id") or "") == agreement_id_value]
    rows = sorted(
        rows,
        key=lambda r: (
            str(r.get("environment_name") or ""),
            str(r.get("workspace_name") or ""),
            str(r.get("notebook_type") or ""),
            str(r.get("notebook_name") or ""),
        ),
    )
    if not rows:
        print("No notebook registry rows found for agreement_id.")
        return

    table_rows = []
    for r in rows:
        url = r.get("notebook_url") or ""
        link = f'<a href="{url}" target="_blank">{r.get("notebook_name") or url}</a>' if url else (r.get("notebook_name") or "(no link)")
        table_rows.append(
            f"<tr><td>{r.get('environment_name','')}</td><td>{r.get('workspace_name','')}</td><td>{r.get('notebook_type','')}</td><td>{link}</td><td>{r.get('related_table','')}</td><td>{r.get('updated_at','')}</td></tr>"
        )
    html = """
    <table>
      <thead><tr><th>Environment</th><th>Workspace</th><th>Type</th><th>Notebook</th><th>Related table</th><th>Updated</th></tr></thead>
      <tbody>{}</tbody>
    </table>
    """.format("".join(table_rows))
    display(HTML(html))

render_notebook_registry(agreement_id)


## Re-run guidance

- Keep `agreement_id` stable for the same agreement.
- This notebook is a governance working notebook: agreement definition, metadata review, business context review, governance review, and human-approved write-back.
- Sandbox, Dev-Test, and Prod reuse approved metadata from this governance control plane.
- DQ enforcement remains in `03_pc` notebooks.
